## Feature Engineering

The goal here is to build a rich customer-level feature table from the 9 raw tables. Every row will represent one customer, and the final output will be a single dataframe ready for modeling.

Features will be grouped into four categories:
- RFM features (recency, frequency, monetary)
- Order behavior features (delivery experience, payment patterns)
- Product features (categories purchased, average product value)
- Review features (satisfaction signals)

In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('../data/olist_orders_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')

# convert dates
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# filter to delivered only
delivered = orders[orders['order_status'] == 'delivered'].copy()

# merge customers
df = delivered.merge(customers, on='customer_id', how='left')

print("Ready:", df.shape)

Ready: (96478, 12)


RFM Explained
RFM is a classic customer analysis framework used by every e-commerce company. It answers three questions about each customer:

Recency — how long ago did they last buy?
A customer with recency = 112 last bought 112 days before the reference date. The higher the number, the longer they've been gone. A churned customer typically has high recency.

Frequency — how many times did they buy?
Almost every customer in this dataset has frequency = 1, which is exactly the problem we're trying to predict — nobody comes back for a second purchase.

Monetary — how much did they spend in total?
Customer 0 spent 141.90 BRL across all their orders. Customer 1 spent only 27.19 BRL. Higher spenders are generally more valuable to retain.

In [2]:
# reference date — the day after the last order in the dataset
reference_date = delivered['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'count'),
    first_order=('order_purchase_timestamp', 'min'),
    last_order=('order_purchase_timestamp', 'max')
).reset_index()

# monetary — total spend per customer
monetary = order_payments.merge(
    delivered[['order_id', 'customer_id']], on='order_id', how='left'
).merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left'
).groupby('customer_unique_id')['payment_value'].sum().reset_index()
monetary.columns = ['customer_unique_id', 'monetary']

rfm = rfm.merge(monetary, on='customer_unique_id', how='left')

print(rfm.shape)
print(rfm.head())

(93358, 6)
                 customer_unique_id  recency  frequency         first_order  \
0  0000366f3b9a7992bf8c76cfdf3221e2      112          1 2018-05-10 10:56:27   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      115          1 2018-05-07 11:11:27   
2  0000f46a3911fa3c0805444483337064      537          1 2017-03-10 21:05:03   
3  0000f6ccb0745a6a4b88665a16c9f078      321          1 2017-10-12 20:29:41   
4  0004aac84e0df4da2b147fca70cf8255      288          1 2017-11-14 19:45:42   

           last_order  monetary  
0 2018-05-10 10:56:27    141.90  
1 2018-05-07 11:11:27     27.19  
2 2017-03-10 21:05:03     86.22  
3 2017-10-12 20:29:41     43.62  
4 2017-11-14 19:45:42    196.89  


In [3]:
# delivery delay — actual vs estimated delivery in days
delivered['delivery_delay'] = (
    delivered['order_delivered_customer_date'] - 
    delivered['order_estimated_delivery_date']
).dt.days

# positive = late, negative = early
customer_delivery = df.merge(
    delivered[['order_id', 'delivery_delay']], on='order_id', how='left'
).groupby('customer_unique_id').agg(
    avg_delivery_delay=('delivery_delay', 'mean'),
    max_delivery_delay=('delivery_delay', 'max'),
    late_deliveries=('delivery_delay', lambda x: (x > 0).sum())
).reset_index()

# payment behavior
payment_behavior = order_payments.merge(
    delivered[['order_id', 'customer_id']], on='order_id', how='left'
).merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left'
).groupby('customer_unique_id').agg(
    avg_installments=('payment_installments', 'mean'),
    unique_payment_types=('payment_type', 'nunique'),
    used_voucher=('payment_type', lambda x: (x == 'voucher').any().astype(int))
).reset_index()

# merge into rfm
rfm = rfm.merge(customer_delivery, on='customer_unique_id', how='left')
rfm = rfm.merge(payment_behavior, on='customer_unique_id', how='left')

print(rfm.shape)
print(rfm.isnull().sum())

(93358, 12)
customer_unique_id      0
recency                 0
frequency               0
first_order             0
last_order              0
monetary                1
avg_delivery_delay      8
max_delivery_delay      8
late_deliveries         0
avg_installments        1
unique_payment_types    1
used_voucher            1
dtype: int64


In [4]:
# fill missing delivery delays with 0 (no delay recorded = assume on time)
rfm['avg_delivery_delay'] = rfm['avg_delivery_delay'].fillna(0)
rfm['max_delivery_delay'] = rfm['max_delivery_delay'].fillna(0)

# fill missing payment and monetary with 0
rfm['monetary'] = rfm['monetary'].fillna(0)
rfm['avg_installments'] = rfm['avg_installments'].fillna(0)
rfm['unique_payment_types'] = rfm['unique_payment_types'].fillna(0)
rfm['used_voucher'] = rfm['used_voucher'].fillna(0)

# confirm no more missing values
print("Missing values:", rfm.isnull().sum().sum())
print("Shape:", rfm.shape)

Missing values: 0
Shape: (93358, 12)


In [5]:
# merge products with order items
order_items_products = order_items.merge(products, on='product_id', how='left')

# merge with delivered orders to get customer
order_items_customers = order_items_products.merge(
    delivered[['order_id', 'customer_id']], on='order_id', how='left'
).merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left'
)

customer_products = order_items_customers.groupby('customer_unique_id').agg(
    avg_item_price=('price', 'mean'),
    max_item_price=('price', 'max'),
    total_items=('order_item_id', 'count'),
    unique_categories=('product_category_name', 'nunique'),
    avg_product_weight=('product_weight_g', 'mean'),
    avg_freight_value=('freight_value', 'mean')
).reset_index()

rfm = rfm.merge(customer_products, on='customer_unique_id', how='left')

print("Shape:", rfm.shape)
print("Missing values:", rfm.isnull().sum().sum())

Shape: (93358, 18)
Missing values: 13


In [6]:
rfm['avg_product_weight'] = rfm['avg_product_weight'].fillna(rfm['avg_product_weight'].median())
rfm['avg_freight_value'] = rfm['avg_freight_value'].fillna(rfm['avg_freight_value'].median())
rfm['unique_categories'] = rfm['unique_categories'].fillna(0)
rfm['avg_item_price'] = rfm['avg_item_price'].fillna(rfm['avg_item_price'].median())
rfm['max_item_price'] = rfm['max_item_price'].fillna(rfm['max_item_price'].median())
rfm['total_items'] = rfm['total_items'].fillna(0)

print("Missing values:", rfm.isnull().sum().sum())

Missing values: 0


In [7]:
# average review score per customer
customer_reviews = order_reviews.merge(
    delivered[['order_id', 'customer_id']], on='order_id', how='left'
).merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left'
).groupby('customer_unique_id').agg(
    avg_review_score=('review_score', 'mean'),
    min_review_score=('review_score', 'min'),
    gave_bad_review=('review_score', lambda x: (x <= 2).any().astype(int))
).reset_index()

rfm = rfm.merge(customer_reviews, on='customer_unique_id', how='left')

# fill missing reviews — customers who never left a review
rfm['avg_review_score'] = rfm['avg_review_score'].fillna(rfm['avg_review_score'].median())
rfm['min_review_score'] = rfm['min_review_score'].fillna(rfm['min_review_score'].median())
rfm['gave_bad_review'] = rfm['gave_bad_review'].fillna(0)

print("Shape:", rfm.shape)
print("Missing values:", rfm.isnull().sum().sum())

Shape: (93358, 21)
Missing values: 0


In [8]:
# rebuild churn labels
customer_orders = df.groupby('customer_unique_id').agg(
    first_order=('order_purchase_timestamp', 'min'),
    total_orders=('order_id', 'count')
).reset_index()

cutoff_date = delivered['order_purchase_timestamp'].max() - pd.DateOffset(months=6)
labeled = customer_orders[customer_orders['first_order'] <= cutoff_date].copy()
labeled['churned'] = (labeled['total_orders'] == 1).astype(int)

# merge churn label into feature table
final = rfm.merge(
    labeled[['customer_unique_id', 'churned']], 
    on='customer_unique_id', 
    how='inner'
)

# drop columns not needed for modeling
final = final.drop(columns=['first_order', 'last_order'])

print("Final dataset shape:", final.shape)
print("\nChurn distribution")
print(final['churned'].value_counts())
print(f"\nChurn rate: {final['churned'].mean():.1%}")

# save
final.to_csv('../data/features.csv', index=False)
print("\nSaved to data/features.csv")

Final dataset shape: (55364, 20)

Churn distribution
churned
1    53161
0     2203
Name: count, dtype: int64

Churn rate: 96.0%

Saved to data/features.csv


## Feature Summary

The final dataset has 55,364 customers and 20 features grouped into four categories:

**RFM features** — recency, frequency, monetary. These capture how recently, how often, and how much each customer spent.

**Delivery features** — average delivery delay, max delivery delay, number of late deliveries. A bad delivery experience is one of the strongest drivers of churn in e-commerce.

**Payment features** — average installments, unique payment types, whether a voucher was used. Customers who pay in many installments or use vouchers may behave differently from regular buyers.

**Product features** — average item price, max item price, total items, unique categories, average product weight, average freight value.

**Review features** — average review score, minimum review score, whether the customer ever left a bad review (score 2 or below).

The churn rate is 96%, meaning the dataset is heavily imbalanced. This will be handled during modeling using class weights rather than oversampling, to avoid introducing artificial patterns into the training data.